# Butyrate Pathway Extraction from Human-GEM

**Obiettivo**: Estrarre reazioni pathway butirrato da Human-GEM per caninizzazione

**Output**: CSV con reazioni, geni (ENSG), GPR rules

---

## Workflow

1. Download Human-GEM model (SBML)
2. Keyword search reazioni correlate a butirrato
3. Estrazione GPR rules e ENSG IDs
4. Classificazione fasi pathway
5. Export CSV per Google Drive

---

## Background Biologico - Pathway Butirrato

**Dal cibo all'energia cellulare:**

1. **Produzione nel lume intestinale**
   - Le fibre alimentari vengono fermentate dai batteri intestinali (microbiota)
   - Questo processo produce butirrato, un acido grasso a catena corta (SCFA)

2. **Trasporto nella cellula**
   - Il butirrato attraversa la membrana cellulare del colonocita
   - Passa dal lume intestinale al citoplasma della cellula

3. **Attivazione metabolica**
   - Il butirrato viene convertito in butirril-CoA
   - Questa forma attivata può entrare nel ciclo di β-ossidazione

4. **β-Ossidazione**
   - Il butirril-CoA viene progressivamente degradato
   - Ogni ciclo rimuove 2 atomi di carbonio producendo acetil-CoA
   - L'acetil-CoA entra nel ciclo di Krebs

5. **Produzione energia**
   - Il ciclo di Krebs e la fosforilazione ossidativa producono ATP
   - Il butirrato fornisce ~70% dell'energia dei colonociti

---

---

## Block 1: Setup e Download Human-GEM

**Cosa fa**:
- Import librerie
- Download Human-GEM da GitHub (release latest)
- Load modello SBML

In [5]:
# Import
import cobra
import pandas as pd
import os
from pathlib import Path
import urllib.request

print(f"✓ COBRApy version: {cobra.__version__}")
print(f"✓ Pandas version: {pd.__version__}")

✓ COBRApy version: 0.29.1
✓ Pandas version: 2.2.3


In [2]:
# Download Human-GEM
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

model_path = data_dir / 'Human-GEM.xml'

if not model_path.exists():
    print("📥 Downloading Human-GEM (latest release)...")
    url = 'https://github.com/SysBioChalmers/Human-GEM/raw/main/model/Human-GEM.xml'
    urllib.request.urlretrieve(url, model_path)
    print(f"✓ Downloaded: {model_path.stat().st_size / 1e6:.1f} MB")
else:
    print(f"✓ Model già scaricato: {model_path}")

📥 Downloading Human-GEM (latest release)...
✓ Downloaded: 43.4 MB


In [6]:
# Load model
print("📂 Loading Human-GEM...")
model = cobra.io.read_sbml_model(str(model_path))

print(f"\n✓ Model loaded:")
print(f"  - Reactions: {len(model.reactions):,}")
print(f"  - Metabolites: {len(model.metabolites):,}")
print(f"  - Genes: {len(model.genes):,}")

# Anteprima prime 5 reazioni
print(f"\n📋 Anteprima prime 5 reazioni:")
print("-" * 80)
for i, rxn in enumerate(model.reactions[:5]):
    print(f"\n{i+1}. {rxn.id} - {rxn.name}")
    print(f"   Equation: {rxn.build_reaction_string()}")
    print(f"   GPR: {rxn.gene_reaction_rule[:60]}..." if len(rxn.gene_reaction_rule) > 60 else f"   GPR: {rxn.gene_reaction_rule}")
    print(f"   Bounds: [{rxn.lower_bound}, {rxn.upper_bound}]")

📂 Loading Human-GEM...

✓ Model loaded:
  - Reactions: 12,971
  - Metabolites: 8,455
  - Genes: 2,887

📋 Anteprima prime 5 reazioni:
--------------------------------------------------------------------------------

1. MAR03905 - ethanol:NAD+ oxidoreductase
   Equation: MAM01796c + MAM02552c --> MAM01249c + MAM02039c + MAM02553c
   GPR: ENSG00000147576 or ENSG00000172955 or ENSG00000180011 or ENS...
   Bounds: [0.0, 1000.0]

2. MAR03907 - Ethanol:NADP+ oxidoreductase
   Equation: MAM01796c + MAM02554c --> MAM01249c + MAM02039c + MAM02555c
   GPR: ENSG00000117448
   Bounds: [0.0, 1000.0]

3. MAR04097 - Acetate:CoA ligase (AMP-forming)
   Equation: MAM01252c + MAM01371c + MAM01597c --> MAM01261c + MAM01334c + MAM02759c
   GPR: ENSG00000131069
   Bounds: [0.0, 1000.0]

4. MAR04099 - Acetate:CoA ligase (AMP-forming)
   Equation: MAM01252m + MAM01371m + MAM01597m --> MAM01261m + MAM01334m + MAM02759m
   GPR: ENSG00000111058 or ENSG00000154930
   Bounds: [0.0, 1000.0]

5. MAR04108 - acetyl ad

---

## Block 2: Keyword Search Reazioni Butirrato

**Modalità:**
- Keyword search (default)
- Lista Reaction_ID specifica (opzionale)

In [24]:
# Keywords per ricerca (modalità default)
keywords = ['but', 'butyr']

# OPZIONALE: Lista specifica di Reaction_ID
# Se popolata, ignora keywords e usa questa lista
reaction_id_list = ["MAR09809","MAR09874","MAR11396","MAR00097","MAR00156","MAR00742","MAR03163","MAR07709","MAR03164","MAR03166","MAR20169","MAR04439"]  # Esempio: ["MAR09809", "MAR00742", "MAR04097"]

print(f"📋 Modalità: {'Lista Reaction_ID' if reaction_id_list else 'Keyword search'}")

📋 Modalità: Lista Reaction_ID


In [25]:
# Cerca reazioni con modalità Reaction_ID o Keyword
import re
found_reactions = []

if reaction_id_list:
    # Modalità: usa lista Reaction_ID
    print(f"🔍 Cerco {len(reaction_id_list)} Reaction_ID specifici...")
    for rxn_id in reaction_id_list:
        try:
            rxn = model.reactions.get_by_id(rxn_id)
            found_reactions.append(rxn)
        except KeyError:
            print(f"⚠️ Reaction_ID non trovato: {rxn_id}")
else:
    # Modalità: usa keyword
    print(f"🔍 Keyword search: {keywords}")
    for rxn in model.reactions:
        rxn_name_lower = rxn.name.lower()
        rxn_id_lower = rxn.id.lower()
        
        for keyword in keywords:
            if re.search(keyword, rxn_name_lower) or re.search(keyword, rxn_id_lower):
                found_reactions.append(rxn)
                break

# Rimuovi duplicati
found_reactions = list(set(found_reactions))

print(f"\n✓ Totale reazioni trovate: {len(found_reactions)}")

🔍 Cerco 12 Reaction_ID specifici...

✓ Totale reazioni trovate: 12


In [26]:
# Anteprima prime 15 reazioni trovate
print("\n📋 ANTEPRIMA PRIME 15 REAZIONI TROVATE")
print("=" * 80)

for i, rxn in enumerate(found_reactions[:15]):
    print(f"\n{i+1}. {rxn.id} - {rxn.name}")
    
    # Equation troncata
    equation = rxn.build_reaction_string()
    equation_display = equation[:70] + "..." if len(equation) > 70 else equation
    print(f"   Equation: {equation_display}")
    
    # GPR troncato
    gpr_display = rxn.gene_reaction_rule[:50] + "..." if len(rxn.gene_reaction_rule) > 50 else rxn.gene_reaction_rule
    print(f"   GPR: {gpr_display}")


📋 ANTEPRIMA PRIME 15 REAZIONI TROVATE

1. MAR09809 - Exchange of butyrate
   Equation: MAM01410e <=> 
   GPR: 

2. MAR03164 - (S)-3-hydroxybutanoyl-CoA hydro-lyase
   Equation: MAM01622m + MAM02040m --> MAM00173m
   GPR: ENSG00000127884

3. MAR03163 - butanoyl-CoA:electron-transfer flavoprotein 2,3-oxidoreductase
   Equation: MAM01412m + MAM01802m --> MAM01622m + MAM01803m
   GPR: ENSG00000117054 or ENSG00000122971 or ENSG00000151...

4. MAR07709 - (3R)-3-hydroxybutanoyl-CoA hydro-lyase
   Equation: MAM00159m <=> MAM01622m + MAM02040m
   GPR: ENSG00000121310

5. MAR20169 - Butyryl-CoA hydrolase
   Equation: MAM01412c + MAM02040c --> MAM01410c + MAM01597c + MAM02039c
   GPR: ENSG00000172497

6. MAR09874 - Butyrate Transport by Smct1
   Equation: MAM01410e + MAM02519e <=> MAM01410c + MAM02519c
   GPR: ENSG00000256870

7. MAR04439 - Transport of (R)-3-Hydroxybutanoyl Coenzyme A into Cytosol
   Equation: MAM00159m <=> MAM00159c
   GPR: ENSG00000155368

8. MAR00156 - Butanoate:CoA ligase (

In [27]:
# Export Block 2 results to CSV
import pandas as pd
from pathlib import Path

# Crea output dir
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Costruisci DataFrame
block2_data = []
for rxn in found_reactions:
    block2_data.append({
        'Reaction_ID': rxn.id,
        'Name': rxn.name,
        'Equation': rxn.build_reaction_string(),
        'GPR_Original': rxn.gene_reaction_rule,
        'Lower_Bound': rxn.lower_bound,
        'Upper_Bound': rxn.upper_bound
    })

df_block2 = pd.DataFrame(block2_data)

# Export
csv_path = output_dir / 'block2_keyword_search_results.csv'
df_block2.to_csv(csv_path, index=False)

print(f"\n✓ Esportato: {csv_path}")
print(f"  Righe: {len(df_block2)}")
print(f"  Dimensione: {csv_path.stat().st_size / 1024:.1f} KB")


✓ Esportato: output/block2_keyword_search_results.csv
  Righe: 12
  Dimensione: 1.8 KB


## Block 3: Estrazione ENSG e Compartimenti

**Cosa fa**:
- Estrae ENSG IDs dai GPR rules
- Identifica compartimenti cellulari delle reazioni
- Costruisce dataset completo con tutte le info

---

### 📚 Cosa sono gli ENSG?

Gli **ENSG** sono **identificatori univoci di geni umani** dal database Ensembl.

**Formato:** `ENSG` + 11 cifre (es: `ENSG00000005187`)

**Nel modello Human-GEM:**
- Ogni reazione metabolica ha un **GPR (Gene-Protein-Reaction) rule**
- Il GPR specifica quali geni umani catalizzano quella reazione
- Questi geni sono identificati con codici **ENSG**

**Esempio reale:**
```
Reazione: MAR00156 (Butanoate:CoA ligase)
GPR_Original: "ENSG00000005187 or ENSG00000066813 or ENSG00000166743"
```
Significa: questa reazione può essere catalizzata da uno qualsiasi di questi 3 geni umani.

---

### 🎯 Perché estraiamo gli ENSG?

**Obiettivo finale:** Caninizzare il modello (sostituire geni umani con geni canini)

**Workflow:**
1. **Block 3:** Estrai ENSG da GPR (geni umani) → lista pulita
2. **BioMart:** Mappa ENSG (umano) → ENSCAFG (cane orthologo)
3. **Phase successiva:** Sostituisci GPR con geni canini

**Esempio mapping:**
- `ENSG00000005187` (gene umano) → `ENSCAFG00000012345` (gene canino)

---

### 🧪 Compartimenti cellulari

I **compartimenti** indicano dove avviene la reazione:
- **c** = cytosol (citoplasma)
- **m** = mitochondria (mitocondrio)
- **e** = extracellular (extracellulare)

**Esempio:**
- Reazione con compartimenti `['c', 'e']` → trasporto attraverso membrana cellulare
- Reazione con compartimenti `['m']` → avviene solo nel mitocondrio

Questa info è utile per capire il pathway completo del butirrato.

---

In [33]:
# Funzione estrazione ENSG da GPR
import re

def extract_ensg_from_gpr(rxn):
    """
    Estrae tutti gli ENSG IDs dal GPR rule di una reazione
    
    Args:
        rxn: oggetto Reaction di COBRApy
    
    Returns:
        list: lista ENSG IDs trovati
    """
    gpr = rxn.gene_reaction_rule
    if not gpr:
        return []
    
    # Pattern per ENSG IDs: ENSG seguito da 11 cifre
    ensg_pattern = r'ENSG\d{11}'
    ensg_list = re.findall(ensg_pattern, gpr)
    
    # Rimuovi duplicati e ordina
    return sorted(list(set(ensg_list)))

# Funzione estrazione compartimenti
def extract_compartments(rxn):
    """Estrae compartimenti da equazione reazione"""
    compartments = set()
    for metabolite in rxn.metabolites:
        # Suffisso metabolite ID (es: MAM01410c -> 'c', MAM01410m -> 'm')
        met_id = metabolite.id
        if len(met_id) > 0:
            # L'ultimo carattere è il compartimento (c=cytosol, m=mitochondria, e=extracellular, ecc)
            comp = met_id[-1]
            compartments.add(comp)
    return sorted(list(compartments))

# Test funzioni
test_rxn = found_reactions[0]
print(f"Test estrazione ENSG:")
print(f"  Reaction: {test_rxn.id}")
print(f"  GPR: {test_rxn.gene_reaction_rule[:70]}..." if len(test_rxn.gene_reaction_rule) > 70 else f"  GPR: {test_rxn.gene_reaction_rule}")
print(f"  ENSG trovati: {extract_ensg_from_gpr(test_rxn)}")

print(f"\nTest compartimenti:")
print(f"  Reaction: {test_rxn.id}")
print(f"  Compartments: {extract_compartments(test_rxn)}")

Test estrazione ENSG:
  Reaction: MAR09809
  GPR: 
  ENSG trovati: []

Test compartimenti:
  Reaction: MAR09809
  Compartments: ['e']


In [45]:
# Anteprima prime righe
print("\n📋 Anteprima prime 5 reazioni:")
print("=" * 100)
df.head()


📋 Anteprima prime 5 reazioni:


,Reaction_ID,Name,Equation,Compartments,ENSG_List,GPR_Original,Lower_Bound,Upper_Bound
0,MAR09809,Exchange of butyrate,MAM01410e <=>,e,,,-1000.0,1000.0
1,MAR03164,(S)-3-hydroxybutanoyl-CoA hydro-lyase,MAM01622m + MAM02040m --> MAM00173m,m,ENSG00000127884,ENSG00000127884,0.0,1000.0
2,MAR03163,"butanoyl-CoA:electron-transfer flavoprotein 2,...",MAM01412m + MAM01802m --> MAM01622m + MAM01803m,m,ENSG00000117054; ENSG00000122971; ENSG00000151...,ENSG00000117054 or ENSG00000122971 or ENSG0000...,0.0,1000.0
3,MAR07709,(3R)-3-hydroxybutanoyl-CoA hydro-lyase,MAM00159m <=> MAM01622m + MAM02040m,m,ENSG00000121310,ENSG00000121310,-1000.0,1000.0
4,MAR20169,Butyryl-CoA hydrolase,MAM01412c + MAM02040c --> MAM01410c + MAM01597...,c,ENSG00000172497,ENSG00000172497,0.0,1000.0


In [46]:
# Costruisci dataset completo
data_rows = []

for rxn in found_reactions:
    ensg_list = extract_ensg_from_gpr(rxn)
    
    data_rows.append({
        'Reaction_ID': rxn.id,
        'Name': rxn.name,
        'Equation': rxn.build_reaction_string(),
        'Compartments': ', '.join(extract_compartments(rxn)),
        'ENSG_List': '; '.join(ensg_list) if ensg_list else '',
        'GPR_Original': rxn.gene_reaction_rule,
        'Lower_Bound': rxn.lower_bound,
        'Upper_Bound': rxn.upper_bound
    })

# Crea DataFrame
df = pd.DataFrame(data_rows)

print(f"\n✓ Dataset costruito: {len(df)} reazioni × {len(df.columns)} colonne")
print(f"\nAnteprima colonne:")
print(df.columns.tolist())


✓ Dataset costruito: 12 reazioni × 8 colonne

Anteprima colonne:
['Reaction_ID', 'Name', 'Equation', 'Compartments', 'ENSG_List', 'GPR_Original', 'Lower_Bound', 'Upper_Bound']


---

## Block 3a: Metadata e Annotazioni Reazioni

**Obiettivo:** Verificare se Human-GEM ha già metadata/subsystems per classificare le reazioni

I modelli metabolici spesso hanno annotazioni come:
- **Subsystem**: categoria pathway (es: "Fatty acid oxidation")
- **Annotation**: link a database (KEGG, Reactome, ecc.)
- **Notes**: descrizioni testuali

---

In [51]:
# Verifica metadata disponibili nelle reazioni
print("📋 METADATA REAZIONI BUTIRRATO")
print("=" * 100)

for i, rxn in enumerate(found_reactions, 1):
    print(f"\n{i}. {rxn.id} - {rxn.name}")
    
    # Subsystem
    if hasattr(rxn, 'subsystem') and rxn.subsystem:
        print(f"   Subsystem: {rxn.subsystem}")
    
    # Annotation
    if hasattr(rxn, 'annotation') and rxn.annotation:
        print(f"   Annotation: {dict(list(rxn.annotation.items())[:3])}...")  # Prime 3
    
    # Notes
    if hasattr(rxn, 'notes') and rxn.notes:
        notes_preview = str(rxn.notes)[:100]
        print(f"   Notes: {notes_preview}...")
    
    # Compartimenti
    print(f"   Compartimenti: {', '.join(extract_compartments(rxn))}")

print("\n" + "=" * 100)
print("📊 SUMMARY SUBSYSTEMS")
print("=" * 100)

subsystems = {}
for rxn in found_reactions:
    if hasattr(rxn, 'subsystem') and rxn.subsystem:
        sub = rxn.subsystem
        if sub not in subsystems:
            subsystems[sub] = []
        subsystems[sub].append(rxn.id)

if subsystems:
    for sub, rxns in subsystems.items():
        print(f"\n{sub}: {len(rxns)} reazioni")
        for rxn_id in rxns:
            print(f"  - {rxn_id}")
else:
    print("\n⚠️ Nessun subsystem trovato nelle reazioni")

📋 METADATA REAZIONI BUTIRRATO

1. MAR09809 - Exchange of butyrate
   Subsystem: Exchange/demand reactions
   Annotation: {'sbo': 'SBO:0000627', 'vmhreaction': 'EX_but[e]'}...
   Notes: {'Confidence Level': '0'}...
   Compartimenti: e

2. MAR03164 - (S)-3-hydroxybutanoyl-CoA hydro-lyase
   Subsystem: Tryptophan metabolism
   Annotation: {'sbo': 'SBO:0000176', 'ec-code': '4.2.1.17', 'kegg.reaction': 'R03026'}...
   Notes: {'Confidence Level': '0', 'AUTHORS': 'PMID:1735445;PMID:8012501;PMID:8188243;PMID:13295248'}...
   Compartimenti: m

3. MAR03163 - butanoyl-CoA:electron-transfer flavoprotein 2,3-oxidoreductase
   Subsystem: Butanoate metabolism
   Annotation: {'sbo': 'SBO:0000176', 'ec-code': '1.3.99.3', 'kegg.reaction': 'R01175'}...
   Notes: {'Confidence Level': '0', 'AUTHORS': 'PMID:13295225;PMID:3597357'}...
   Compartimenti: m

4. MAR07709 - (3R)-3-hydroxybutanoyl-CoA hydro-lyase
   Subsystem: Butanoate metabolism
   Annotation: {'sbo': 'SBO:0000176', 'ec-code': '4.2.1.55', 'kegg.

---

## Block 3b: Classificazione Fasi Pathway

**Obiettivo:** Verificare che le 12 reazioni coprano l'intero pathway butirrato fino ad acetil-CoA

**Fasi attese:**
1. **Trasporto** (extracellulare → citoplasma → mitocondrio)
2. **Attivazione** (butirrato → butirril-CoA)
3. **β-Ossidazione** (butirril-CoA → acetil-CoA)
4. **Opzionali** (idrolasi, shuttle)

**Domanda critica:** La β-ossidazione si ferma ad **acetil-CoA**? 
- Se SÌ → Krebs è nel GEM generale (non va caninizzato come "butirrato")
- Se NO → Mancano reazioni

---

In [48]:
# Classificazione automatica reazioni per fase
import re

def classify_reaction_phase(rxn):
    """Classifica reazione in base a nome e compartimenti"""
    name_lower = rxn.name.lower()
    comps = extract_compartments(rxn)
    equation = rxn.build_reaction_string().lower()
    
    # Trasporto
    if 'transport' in name_lower or 'exchange' in name_lower:
        return "TRASPORTO"
    
    # Attivazione (ligasi che produce butirril-CoA)
    if 'ligase' in name_lower and 'butanoate' in name_lower:
        return "ATTIVAZIONE"
    
    # β-Ossidazione (reazioni su butirril-CoA o derivati)
    if any(x in name_lower for x in ['oxidoreductase', 'hydro-lyase']):
        if any(x in name_lower for x in ['butanoyl', 'hydroxybutanoyl']):
            return "β-OSSIDAZIONE"
    
    # Idrolasi (opzionale)
    if 'hydrolase' in name_lower:
        return "OPZIONALE"
    
    return "ALTRO"

# Classifica tutte le reazioni
print("📊 CLASSIFICAZIONE FASI PATHWAY BUTIRRATO")
print("=" * 100)

phases = {}
for rxn in found_reactions:
    phase = classify_reaction_phase(rxn)
    if phase not in phases:
        phases[phase] = []
    phases[phase].append(rxn)

# Mostra per fase
for phase in ["TRASPORTO", "ATTIVAZIONE", "β-OSSIDAZIONE", "OPZIONALE", "ALTRO"]:
    if phase in phases:
        print(f"\n{'='*100}")
        print(f"FASE: {phase} ({len(phases[phase])} reazioni)")
        print('='*100)
        
        for rxn in phases[phase]:
            print(f"\n{rxn.id} - {rxn.name}")
            print(f"  Compartimenti: {', '.join(extract_compartments(rxn))}")
            print(f"  Equazione: {rxn.build_reaction_string()}")
            print(f"  Bounds: [{rxn.lower_bound}, {rxn.upper_bound}]")

📊 CLASSIFICAZIONE FASI PATHWAY BUTIRRATO

FASE: TRASPORTO (5 reazioni)

MAR09809 - Exchange of butyrate
  Compartimenti: e
  Equazione: MAM01410e <=> 
  Bounds: [-1000.0, 1000.0]

MAR09874 - Butyrate Transport by Smct1
  Compartimenti: c, e
  Equazione: MAM01410e + MAM02519e <=> MAM01410c + MAM02519c
  Bounds: [-1000.0, 1000.0]

MAR04439 - Transport of (R)-3-Hydroxybutanoyl Coenzyme A into Cytosol
  Compartimenti: c, m
  Equazione: MAM00159m <=> MAM00159c
  Bounds: [-1000.0, 1000.0]

MAR11396 - Butyrate Transport via Proton Symport, Reversible
  Compartimenti: c, e
  Equazione: MAM01410e + MAM02039e <=> MAM01410c + MAM02039c
  Bounds: [-1000.0, 1000.0]

MAR00097 - Butyrate Mitochondrial Transport via Proton Symport, Reversible
  Compartimenti: c, m
  Equazione: MAM01410c + MAM02039c <=> MAM01410m + MAM02039m
  Bounds: [-1000.0, 1000.0]

FASE: ATTIVAZIONE (2 reazioni)

MAR00156 - Butanoate:CoA ligase (AMP-forming)
  Compartimenti: c
  Equazione: MAM01371c + MAM01410c + MAM01597c --> MAM

In [49]:
# Verifica prodotti finali β-ossidazione
print("\n" + "="*100)
print("🔬 VERIFICA: Prodotto finale β-ossidazione")
print("="*100)

# Cerca reazioni β-ossidazione
beta_ox_reactions = phases.get("β-OSSIDAZIONE", [])

print(f"\nReazioni β-ossidazione trovate: {len(beta_ox_reactions)}")

# Analizza prodotti
all_products = set()
for rxn in beta_ox_reactions:
    equation = rxn.build_reaction_string()
    print(f"\n{rxn.id}: {rxn.name}")
    print(f"  {equation}")
    
    # Estrai metaboliti prodotti (lato destro equazione)
    if '-->' in equation:
        products_side = equation.split('-->')[1]
    elif '<=>' in equation:
        products_side = equation.split('<=>')[1]
    else:
        products_side = equation
    
    # Cerca pattern acetil-CoA o acetyl nei prodotti
    if 'acet' in products_side.lower():
        print(f"  ⚠️ Possibile acetil-CoA nei prodotti!")
        all_products.add(rxn.id)

print(f"\n{'='*100}")
print(f"CONCLUSIONE:")
if all_products:
    print(f"✅ Trovate reazioni che producono acetil-CoA")
    print(f"   → Il pathway butirrato si ferma ad acetil-CoA")
    print(f"   → Krebs è nel GEM generale (non va caninizzato)")
else:
    print(f"⚠️ Nessuna reazione produce esplicitamente acetil-CoA")
    print(f"   → Verificare con Daniela se mancano reazioni")
print("="*100)


🔬 VERIFICA: Prodotto finale β-ossidazione

Reazioni β-ossidazione trovate: 4

MAR03164: (S)-3-hydroxybutanoyl-CoA hydro-lyase
  MAM01622m + MAM02040m --> MAM00173m

MAR03163: butanoyl-CoA:electron-transfer flavoprotein 2,3-oxidoreductase
  MAM01412m + MAM01802m --> MAM01622m + MAM01803m

MAR07709: (3R)-3-hydroxybutanoyl-CoA hydro-lyase
  MAM00159m <=> MAM01622m + MAM02040m

MAR03166: (S)-3-Hydroxybutanoyl-CoA:NAD+ oxidoreductase
  MAM00173m + MAM02552m --> MAM01255m + MAM02039m + MAM02553m

CONCLUSIONE:
⚠️ Nessuna reazione produce esplicitamente acetil-CoA
   → Verificare con Daniela se mancano reazioni


In [50]:
# Classificazione automatica reazioni per fase
import re

def classify_reaction_phase(rxn):
    """Classifica reazione in base a nome e compartimenti"""
    name_lower = rxn.name.lower()
    comps = extract_compartments(rxn)
    equation = rxn.build_reaction_string().lower()
    
    # Trasporto
    if 'transport' in name_lower or 'exchange' in name_lower:
        return "TRASPORTO"
    
    # Attivazione (ligasi che produce butirril-CoA)
    if 'ligase' in name_lower and 'butanoate' in name_lower:
        return "ATTIVAZIONE"
    
    # β-Ossidazione (reazioni su butirril-CoA o derivati)
    if any(x in name_lower for x in ['oxidoreductase', 'hydro-lyase']):
        if any(x in name_lower for x in ['butanoyl', 'hydroxybutanoyl']):
            return "β-OSSIDAZIONE"
    
    # Idrolasi (opzionale)
    if 'hydrolase' in name_lower:
        return "OPZIONALE"
    
    return "ALTRO"

# Classifica tutte le reazioni
print("📊 CLASSIFICAZIONE FASI PATHWAY BUTIRRATO")
print("=" * 100)

phases = {}
for rxn in found_reactions:
    phase = classify_reaction_phase(rxn)
    if phase not in phases:
        phases[phase] = []
    phases[phase].append(rxn)

# Mostra per fase
for phase in ["TRASPORTO", "ATTIVAZIONE", "β-OSSIDAZIONE", "OPZIONALE", "ALTRO"]:
    if phase in phases:
        print(f"\n{'='*100}")
        print(f"FASE: {phase} ({len(phases[phase])} reazioni)")
        print('='*100)
        
        for rxn in phases[phase]:
            print(f"\n{rxn.id} - {rxn.name}")
            print(f"  Compartimenti: {', '.join(extract_compartments(rxn))}")
            print(f"  Equazione: {rxn.build_reaction_string()}")
            print(f"  Bounds: [{rxn.lower_bound}, {rxn.upper_bound}]")

📊 CLASSIFICAZIONE FASI PATHWAY BUTIRRATO

FASE: TRASPORTO (5 reazioni)

MAR09809 - Exchange of butyrate
  Compartimenti: e
  Equazione: MAM01410e <=> 
  Bounds: [-1000.0, 1000.0]

MAR09874 - Butyrate Transport by Smct1
  Compartimenti: c, e
  Equazione: MAM01410e + MAM02519e <=> MAM01410c + MAM02519c
  Bounds: [-1000.0, 1000.0]

MAR04439 - Transport of (R)-3-Hydroxybutanoyl Coenzyme A into Cytosol
  Compartimenti: c, m
  Equazione: MAM00159m <=> MAM00159c
  Bounds: [-1000.0, 1000.0]

MAR11396 - Butyrate Transport via Proton Symport, Reversible
  Compartimenti: c, e
  Equazione: MAM01410e + MAM02039e <=> MAM01410c + MAM02039c
  Bounds: [-1000.0, 1000.0]

MAR00097 - Butyrate Mitochondrial Transport via Proton Symport, Reversible
  Compartimenti: c, m
  Equazione: MAM01410c + MAM02039c <=> MAM01410m + MAM02039m
  Bounds: [-1000.0, 1000.0]

FASE: ATTIVAZIONE (2 reazioni)

MAR00156 - Butanoate:CoA ligase (AMP-forming)
  Compartimenti: c
  Equazione: MAM01371c + MAM01410c + MAM01597c --> MAM

## Block 4: Anteprima Dati

**Cosa fa**:
- Mostra statistiche dataset
- Anteprima prime righe
- Conta geni unici

In [36]:
# Anteprima prime 10 reazioni
print("\n📋 ANTEPRIMA PRIME 10 REAZIONI")
print("=" * 50)

# Mostra colonne principali
display_cols = ['Reaction_ID', 'Name', 'ENSG_List']
df[display_cols].head(10)


📋 ANTEPRIMA PRIME 10 REAZIONI


,Reaction_ID,Name,ENSG_List
0,MAR09809,Exchange of butyrate,
1,MAR03164,(S)-3-hydroxybutanoyl-CoA hydro-lyase,ENSG00000127884
2,MAR03163,"butanoyl-CoA:electron-transfer flavoprotein 2,...",ENSG00000117054; ENSG00000122971; ENSG00000151...
3,MAR07709,(3R)-3-hydroxybutanoyl-CoA hydro-lyase,ENSG00000121310
4,MAR20169,Butyryl-CoA hydrolase,ENSG00000172497
5,MAR09874,Butyrate Transport by Smct1,ENSG00000256870
6,MAR04439,Transport of (R)-3-Hydroxybutanoyl Coenzyme A ...,ENSG00000155368
7,MAR00156,Butanoate:CoA ligase (AMP-forming),ENSG00000005187; ENSG00000066813; ENSG00000166...
8,MAR00742,"Fatty-Acid- Coenzyme A Ligase (Butanoate), Mit...",ENSG00000166743
9,MAR03166,(S)-3-Hydroxybutanoyl-CoA:NAD+ oxidoreductase,ENSG00000072506; ENSG00000084754; ENSG00000138796


In [37]:
# Esempi reazioni
print("\n🔬 ESEMPI REAZIONI")
print("=" * 50)

for i, row in df.head(3).iterrows():
    print(f"\n{i+1}. {row['Reaction_ID']}")
    print(f"   Nome: {row['Name']}")
    print(f"   ENSG: {row['ENSG_List'][:50]}..." if len(row['ENSG_List']) > 50 else f"   ENSG: {row['ENSG_List']}")
    print(f"   Compartments: {row['Compartments']}")


🔬 ESEMPI REAZIONI

1. MAR09809
   Nome: Exchange of butyrate
   ENSG: 
   Compartments: e

2. MAR03164
   Nome: (S)-3-hydroxybutanoyl-CoA hydro-lyase
   ENSG: ENSG00000127884
   Compartments: m

3. MAR03163
   Nome: butanoyl-CoA:electron-transfer flavoprotein 2,3-oxidoreductase
   ENSG: ENSG00000117054; ENSG00000122971; ENSG00000151498;...
   Compartments: m


## Block 5: Export CSV per Google Drive

**Cosa fa**:
- Salva dataset in CSV
- Crea file separato con solo lista ENSG unici (per BioMart)
- Mostra path file generati

In [42]:
# Crea output directory
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Export reazioni complete
csv_path = output_dir / 'butyrate_reactions.csv'
df.to_csv(csv_path, index=False)

print(f"✓ Esportato: {csv_path}")
print(f"  Righe: {len(df)}")
print(f"  Dimensione: {csv_path.stat().st_size / 1024:.1f} KB")

✓ Esportato: output/butyrate_reactions.csv
  Righe: 12
  Dimensione: 2.2 KB


In [43]:
# Export lista ENSG unici (per BioMart)
ensg_df = pd.DataFrame({
    'ENSG_ID': sorted(list(all_ensg))
})

ensg_csv_path = output_dir / 'ensg_list_for_biomart.csv'
ensg_df.to_csv(ensg_csv_path, index=False)

print(f"\n✓ Esportato: {ensg_csv_path}")
print(f"  ENSG unici: {len(ensg_df)}")


✓ Esportato: output/ensg_list_for_biomart.csv
  ENSG unici: 21


In [44]:
# Summary finale
print("\n" + "=" * 50)
print("✅ EXPORT COMPLETATO")
print("=" * 50)
print(f"\nFile generati:")
print(f"1. {csv_path}")
print(f"   → Importa in Google Drive/Sheets")
print(f"\n2. {ensg_csv_path}")
print(f"   → Usa per BioMart (Human→Dog orthologs)")
print(f"\nProssimi step:")
print(f"  1. Upload CSV su Google Drive")
print(f"  2. BioMart: mappa ENSG umani → ENSCAFG canini")
print(f"  3. Sostituisci GPR con geni canini")


✅ EXPORT COMPLETATO

File generati:
1. output/butyrate_reactions.csv
   → Importa in Google Drive/Sheets

2. output/ensg_list_for_biomart.csv
   → Usa per BioMart (Human→Dog orthologs)

Prossimi step:
  1. Upload CSV su Google Drive
  2. BioMart: mappa ENSG umani → ENSCAFG canini
  3. Sostituisci GPR con geni canini


# Setup FBA: simula uptake butirrato
# Nota: Lavoriamo direttamente sul modello (COBRApy 0.29.1 ha bug con copy() su Python 3.14)

print("🔬 SETUP SIMULAZIONE FBA")
print("=" * 100)

# 1. Salva bounds originali per eventuale ripristino
original_butyrate_bounds = model.reactions.get_by_id("MAR09809").bounds

# 2. Imposta uptake butirrato
butyrate_exchange = model.reactions.get_by_id("MAR09809")  # Exchange of butyrate
print(f"\n1. Uptake butirrato:")
print(f"   Reazione: {butyrate_exchange.id} - {butyrate_exchange.name}")
print(f"   Bounds originali: {original_butyrate_bounds}")

# Imposta uptake a 1 mmol/gDW/h
butyrate_exchange.lower_bound = -1.0  # Uptake
print(f"   Bounds FBA: [{butyrate_exchange.lower_bound}, {butyrate_exchange.upper_bound}]")

# 3. Verifica funzione obiettivo
print(f"\n2. Funzione obiettivo:")
print(f"   {model.objective.expression}")

# 4. Stato modello
print(f"\n3. Stato modello:")
print(f"   Reazioni totali: {len(model.reactions):,}")
print(f"   Metaboliti: {len(model.metabolites):,}")
print(f"   Geni: {len(model.genes):,}")

print("\n" + "=" * 100)

In [53]:
# Esegui FBA
print("⚡ ESECUZIONE FBA")
print("=" * 100)

try:
    solution = model.optimize()
    
    print(f"\nStatus: {solution.status}")
    print(f"Objective value (biomass): {solution.objective_value:.6f}")
    
    if solution.status == 'optimal':
        print("\n✅ Soluzione ottimale trovata!")
        print(f"   → Il modello può metabolizzare il butirrato")
    else:
        print(f"\n⚠️ Soluzione non ottimale: {solution.status}")
        
except Exception as e:
    print(f"\n❌ Errore durante FBA: {e}")
    solution = None

print("=" * 100)

⚡ ESECUZIONE FBA

Status: optimal
Objective value (biomass): 124.868148

✅ Soluzione ottimale trovata!
   → Il modello può metabolizzare il butirrato


In [54]:
# Analizza flussi delle 12 reazioni butirrato
if solution and solution.status == 'optimal':
    print("\n📊 FLUSSI REAZIONI BUTIRRATO (12 reazioni)")
    print("=" * 100)
    
    butyrate_rxn_ids = [rxn.id for rxn in found_reactions]
    active_butyrate_rxns = []
    
    for rxn_id in butyrate_rxn_ids:
        flux = solution.fluxes[rxn_id]
        rxn = model.reactions.get_by_id(rxn_id)
        
        if abs(flux) > 1e-6:  # Reazione attiva (flusso > 0)
            active_butyrate_rxns.append((rxn_id, flux))
            print(f"\n✓ {rxn_id} - {rxn.name}")
            print(f"  Flusso: {flux:.6f} mmol/gDW/h")
            print(f"  Subsystem: {rxn.subsystem if hasattr(rxn, 'subsystem') else 'N/A'}")
        else:
            print(f"\n✗ {rxn_id} - {rxn.name}")
            print(f"  Flusso: {flux:.6f} (INATTIVA)")
    
    print(f"\n{'='*100}")
    print(f"SUMMARY: {len(active_butyrate_rxns)}/12 reazioni attive nel pathway")
    print("=" * 100)
else:
    print("\n⚠️ Nessuna soluzione FBA disponibile per analisi flussi")


📊 FLUSSI REAZIONI BUTIRRATO (12 reazioni)

✗ MAR09809 - Exchange of butyrate
  Flusso: 0.000000 (INATTIVA)

✗ MAR03164 - (S)-3-hydroxybutanoyl-CoA hydro-lyase
  Flusso: 0.000000 (INATTIVA)

✗ MAR03163 - butanoyl-CoA:electron-transfer flavoprotein 2,3-oxidoreductase
  Flusso: 0.000000 (INATTIVA)

✗ MAR07709 - (3R)-3-hydroxybutanoyl-CoA hydro-lyase
  Flusso: 0.000000 (INATTIVA)

✗ MAR20169 - Butyryl-CoA hydrolase
  Flusso: 0.000000 (INATTIVA)

✗ MAR09874 - Butyrate Transport by Smct1
  Flusso: 0.000000 (INATTIVA)

✗ MAR04439 - Transport of (R)-3-Hydroxybutanoyl Coenzyme A into Cytosol
  Flusso: 0.000000 (INATTIVA)

✗ MAR00156 - Butanoate:CoA ligase (AMP-forming)
  Flusso: 0.000000 (INATTIVA)

✗ MAR00742 - Fatty-Acid- Coenzyme A Ligase (Butanoate), Mitochondrial
  Flusso: 0.000000 (INATTIVA)

✗ MAR03166 - (S)-3-Hydroxybutanoyl-CoA:NAD+ oxidoreductase
  Flusso: 0.000000 (INATTIVA)

✗ MAR11396 - Butyrate Transport via Proton Symport, Reversible
  Flusso: 0.000000 (INATTIVA)

✗ MAR00097 - B

In [56]:
# DOMANDA CRITICA: Cerca reazioni NON nelle 12 che processano butirril-CoA o producono acetil-CoA
if solution and solution.status == 'optimal':
    print("\n🔍 REAZIONI EXTRA-PATHWAY (non nelle 12) ATTIVE")
    print("=" * 100)
    print("\nCerco reazioni che:")
    print("  1. Processano butirril-CoA o derivati")
    print("  2. Producono acetil-CoA")
    print("  3. NON sono nelle 12 reazioni di Daniela")
    
    butyrate_rxn_ids_set = set([rxn.id for rxn in found_reactions])
    extra_reactions = []
    
    # Cerca reazioni attive fuori dalle 12
    for rxn_id, flux in solution.fluxes.items():
        if abs(flux) > 1e-6 and rxn_id not in butyrate_rxn_ids_set:
            rxn = model.reactions.get_by_id(rxn_id)
            equation = rxn.build_reaction_string().lower()
            
            # Filtra per reazioni rilevanti al butirrato
            if any(keyword in equation for keyword in ['butanoyl', 'butyryl', 'hydroxybutanoyl', 'acetyl-coa', 'mam01261']):
                extra_reactions.append({
                    'id': rxn_id,
                    'name': rxn.name,
                    'flux': flux,
                    'equation': rxn.build_reaction_string(),
                    'subsystem': rxn.subsystem if hasattr(rxn, 'subsystem') else 'N/A'
                })
    
    if extra_reactions:
        print(f"\n⚠️ TROVATE {len(extra_reactions)} REAZIONI EXTRA-PATHWAY ATTIVE:")
        print("=" * 100)
        
        for i, rxn_data in enumerate(extra_reactions[:10], 1):  # Mostra prime 10
            print(f"\n{i}. {rxn_data['id']} - {rxn_data['name']}")
            print(f"   Flusso: {rxn_data['flux']:.6f}")
            print(f"   Subsystem: {rxn_data['subsystem']}")
            print(f"   Equazione: {rxn_data['equation'][:80]}...")
        
        if len(extra_reactions) > 10:
            print(f"\n... e altre {len(extra_reactions) - 10} reazioni")
        
        print(f"\n{'='*100}")
        print("CONCLUSIONE:")
        print("⚠️ Il modello usa reazioni FUORI dalle 12 per completare il pathway!")
        print("   → Probabilmente manca una reazione nelle 12 (es: tiolasi per acetil-CoA)")
        print("=" * 100)
    else:
        print(f"\n✅ NESSUNA reazione extra-pathway trovata")
        print("   → Le 12 reazioni sembrano sufficienti")
        print("=" * 100)
else:
    print("\n⚠️ Nessuna soluzione FBA disponibile")

# Ripristina bounds originali (se definiti)
try:
    model.reactions.get_by_id("MAR09809").bounds = original_butyrate_bounds
    print(f"\n✓ Bounds butirrato ripristinati: {original_butyrate_bounds}")
except NameError:
    print("\n⚠️ Bounds non ripristinati (esegui prima la cella setup FBA)")


🔍 REAZIONI EXTRA-PATHWAY (non nelle 12) ATTIVE

Cerco reazioni che:
  1. Processano butirril-CoA o derivati
  2. Producono acetil-CoA
  3. NON sono nelle 12 reazioni di Daniela

⚠️ TROVATE 34 REAZIONI EXTRA-PATHWAY ATTIVE:

1. MAR04604 - acetyl-CoA:acetoacetyl-CoA C-acetyltransferase (thioester-hydrolysing, carboxymethyl-forming)
   Flusso: 15.742066
   Subsystem: Butanoate metabolism
   Equazione: MAM01255x + MAM01261x + MAM02040x --> MAM01597x + MAM02039x + MAM02131x...

2. MAR04145 - acetyl-CoA:oxaloacetate C-acetyltransferase (thioester-hydrolysing)
   Flusso: 27.787561
   Subsystem: Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism
   Equazione: MAM01261m + MAM02040m + MAM02633m --> MAM01587m + MAM01597m + MAM02039m...

3. MAR03077 - 
   Flusso: 0.012945
   Subsystem: Beta oxidation of even-chain fatty acids (peroxisomal)
   Equazione: MAM00890x + MAM01597x --> MAM01261x + MAM02678x...

4. MAR03106 - 
   Flusso: -6.690507
   Subsystem: Transport reactions
   Equazi

---

### ⚠️ Problema: Il modello non usa il butirrato!

FBA ha ottimizzato biomassa usando altre fonti di carbonio (glucosio, palmitato, ecc.).
Le 12 reazioni butirrato hanno flusso 0.

**Soluzione:** FORZARE l'uso del butirrato come UNICA fonte di carbonio.

---

In [57]:
# RETRY FBA: Forza uso butirrato chiudendo altre fonti carbonio
print("🔧 RETRY FBA - BUTIRRATO COME UNICA FONTE CARBONIO")
print("=" * 100)

# 1. Chiudi tutte le exchange reactions (fonti esterne)
print("\n1. Chiudo tutte le fonti di carbonio esterne...")
exchange_rxns_closed = []
for rxn in model.exchanges:
    if rxn.id != "MAR09809":  # Non chiudere il butirrato!
        if rxn.lower_bound < 0:  # È un uptake
            exchange_rxns_closed.append((rxn.id, rxn.lower_bound))
            rxn.lower_bound = 0  # Chiudi uptake

print(f"   Chiuse {len(exchange_rxns_closed)} exchange reactions")

# 2. Forza uptake butirrato
butyrate_exchange = model.reactions.get_by_id("MAR09809")
butyrate_exchange.lower_bound = -1.0
print(f"\n2. Butirrato uptake forzato: {butyrate_exchange.lower_bound}")

# 3. Esegui FBA
print(f"\n3. Eseguo FBA con SOLO butirrato...")
try:
    solution2 = model.optimize()
    print(f"   Status: {solution2.status}")
    print(f"   Objective: {solution2.objective_value:.6f}")
except Exception as e:
    print(f"   ❌ Errore: {e}")
    solution2 = None

print("=" * 100)

🔧 RETRY FBA - BUTIRRATO COME UNICA FONTE CARBONIO

1. Chiudo tutte le fonti di carbonio esterne...
   Chiuse 1659 exchange reactions

2. Butirrato uptake forzato: -1.0

3. Eseguo FBA con SOLO butirrato...
   Status: optimal
   Objective: -0.000000


In [58]:
# Analizza flussi RETRY
if solution2 and solution2.status == 'optimal':
    print("\n📊 FLUSSI REAZIONI BUTIRRATO (con uptake forzato)")
    print("=" * 100)
    
    active_count = 0
    for rxn_id in [r.id for r in found_reactions]:
        flux = solution2.fluxes[rxn_id]
        rxn = model.reactions.get_by_id(rxn_id)
        
        if abs(flux) > 1e-6:
            active_count += 1
            print(f"\n✓ {rxn_id} - {rxn.name}")
            print(f"  Flusso: {flux:.6f}")
        else:
            print(f"\n✗ {rxn_id} - Flusso: 0")
    
    print(f"\n{'='*100}")
    print(f"SUMMARY: {active_count}/12 reazioni attive")
    
    if active_count > 0:
        print("✅ Il pathway butirrato È USATO quando forzato!")
    else:
        print("⚠️ Anche forzando, il pathway NON si attiva (problema strutturale)")
    print("=" * 100)
else:
    print("\n❌ FBA fallita - Il modello non può usare SOLO butirrato")
    print("   → Probabilmente manca una reazione critica nel pathway")

# Ripristina exchange reactions
print(f"\n🔄 Ripristino {len(exchange_rxns_closed)} exchange reactions...")
for rxn_id, original_lb in exchange_rxns_closed:
    model.reactions.get_by_id(rxn_id).lower_bound = original_lb
print("✓ Exchange reactions ripristinate")


📊 FLUSSI REAZIONI BUTIRRATO (con uptake forzato)

✗ MAR09809 - Flusso: 0

✗ MAR03164 - Flusso: 0

✗ MAR03163 - Flusso: 0

✗ MAR07709 - Flusso: 0

✗ MAR20169 - Flusso: 0

✗ MAR09874 - Flusso: 0

✗ MAR04439 - Flusso: 0

✗ MAR00156 - Flusso: 0

✗ MAR00742 - Flusso: 0

✗ MAR03166 - Flusso: 0

✗ MAR11396 - Flusso: 0

✗ MAR00097 - Flusso: 0

SUMMARY: 0/12 reazioni attive
⚠️ Anche forzando, il pathway NON si attiva (problema strutturale)

🔄 Ripristino 1659 exchange reactions...
✓ Exchange reactions ripristinate


In [ ]:
# DOMANDA CRITICA: Cerca reazioni NON nelle 12 che processano butirril-CoA o producono acetil-CoA
if solution and solution.status == 'optimal':
    print("\n🔍 REAZIONI EXTRA-PATHWAY (non nelle 12) ATTIVE")
    print("=" * 100)
    print("\nCerco reazioni che:")
    print("  1. Processano butirril-CoA o derivati")
    print("  2. Producono acetil-CoA")
    print("  3. NON sono nelle 12 reazioni di Daniela")
    
    butyrate_rxn_ids_set = set([rxn.id for rxn in found_reactions])
    extra_reactions = []
    
    # Cerca reazioni attive fuori dalle 12
    for rxn_id, flux in solution.fluxes.items():
        if abs(flux) > 1e-6 and rxn_id not in butyrate_rxn_ids_set:
            rxn = model_fba.reactions.get_by_id(rxn_id)
            equation = rxn.build_reaction_string().lower()
            
            # Filtra per reazioni rilevanti al butirrato
            if any(keyword in equation for keyword in ['butanoyl', 'butyryl', 'hydroxybutanoyl', 'acetyl-coa', 'mam01261']):
                extra_reactions.append({
                    'id': rxn_id,
                    'name': rxn.name,
                    'flux': flux,
                    'equation': rxn.build_reaction_string(),
                    'subsystem': rxn.subsystem if hasattr(rxn, 'subsystem') else 'N/A'
                })
    
    if extra_reactions:
        print(f"\n⚠️ TROVATE {len(extra_reactions)} REAZIONI EXTRA-PATHWAY ATTIVE:")
        print("=" * 100)
        
        for i, rxn_data in enumerate(extra_reactions[:10], 1):  # Mostra prime 10
            print(f"\n{i}. {rxn_data['id']} - {rxn_data['name']}")
            print(f"   Flusso: {rxn_data['flux']:.6f}")
            print(f"   Subsystem: {rxn_data['subsystem']}")
            print(f"   Equazione: {rxn_data['equation'][:80]}...")
        
        if len(extra_reactions) > 10:
            print(f"\n... e altre {len(extra_reactions) - 10} reazioni")
        
        print(f"\n{'='*100}")
        print("CONCLUSIONE:")
        print("⚠️ Il modello usa reazioni FUORI dalle 12 per completare il pathway!")
        print("   → Probabilmente manca una reazione nelle 12 (es: tiolasi per acetil-CoA)")
        print("=" * 100)
    else:
        print(f"\n✅ NESSUNA reazione extra-pathway trovata")
        print("   → Le 12 reazioni sembrano sufficienti")
        print("=" * 100)
else:
    print("\n⚠️ Nessuna soluzione FBA disponibile")